<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student_attempt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

First I will start with checking for any missing data in the 3 tables. I will ask AI to generate code to check for the NA. I will specify also on the format of the output that I want it to be the list of columns and the count of NA

In [ ]:
for name, df in tables.items():
    print(f"=== NA Counts in '{name}' table ===")
    print(df.isna().sum())
    print()

=== NA Counts in 'trips' table ===
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

=== NA Counts in 'bikes' table ===
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

=== NA Counts in 'stations' table ===
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64



There are some missing data in trips specifically in duration_min of 30 records and 1 NA in table stations for 1 records in column capacity.
Next I will be checking for any data duplications in the 3 tables.

In [ ]:
for name, df in tables.items():
    print(f"=== Duplication Check in '{name}' table ===")
    # Check for completely duplicate rows
    row_dupes = df.duplicated().sum()
    print(f"Total complete duplicate rows: {row_dupes}")

    # Determine the primary ID column
    id_col = 'trip_id' if name == 'trips' else ('bike_id' if name == 'bikes' else 'station_id')
    if id_col in df.columns:
        id_dupes = df.duplicated(subset=[id_col]).sum()
        print(f"Duplicate values in key '{id_col}': {id_dupes}")
    print()

=== Duplication Check in 'trips' table ===
Total complete duplicate rows: 12
Duplicate values in key 'trip_id': 12

=== Duplication Check in 'bikes' table ===
Total complete duplicate rows: 0
Duplicate values in key 'bike_id': 0

=== Duplication Check in 'stations' table ===
Total complete duplicate rows: 0
Duplicate values in key 'station_id': 0



There are 12 exact duplication in the table trips.
I want to see in more detail to verify the nature of the duplication so im asking AI to create another script to show the value of the duplication

In [ ]:
import pandas as pd
from IPython.display import display

# Filter and display duplicate rows in 'trips'
trips_df = tables['trips']
duplicates_trips = trips_df[trips_df.duplicated(keep=False)].sort_values(by='trip_id')

print(f"Found {duplicates_trips.shape[0]} rows that are part of complete duplicates (including originals):")
display(duplicates_trips)

Found 24 rows that are part of complete duplicates (including originals):


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member


So i have verify the the duplication detexted by AI is the exact duplication.
Next I want to check the consistency of the data like the formating. I think I will go by each table so I will start with the table bikes first.

In [ ]:
print("=== Checking 'bikes' Table Consistency ===")
bikes_df = tables['bikes']

# 1. Check data types and structure
print("\n--- Data Types & Info ---")
print(bikes_df.info())

# 2. Check unique values and categories for bike_type
print("\n--- Unique Bike Types ---")
print(bikes_df['bike_type'].value_counts(dropna=False))

# 3. Check distribution and ranges for purchase_year
print("\n--- Purchase Year Distribution ---")
print(bikes_df['purchase_year'].describe())
print("\nUnique Purchase Years:")
print(sorted(bikes_df['purchase_year'].unique()))

# 4. Check bike_id formatting (e.g., matching a pattern like 'B' followed by numbers)
print("\n--- ID Formatting Check ---")
non_conforming_ids = bikes_df[~bikes_df['bike_id'].astype(str).str.match(r'^B\d+$')]
if non_conforming_ids.empty:
    print("All bike_id values conform to the standard 'Bxxxx' format.")
else:
    print("Found non-conforming bike_ids:")
    display(non_conforming_ids)

=== Checking 'bikes' Table Consistency ===

--- Data Types & Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   bike_id        60 non-null     object
 1   bike_type      60 non-null     object
 2   purchase_year  60 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 1.5+ KB
None

--- Unique Bike Types ---
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64

--- Purchase Year Distribution ---
count      60.000000
mean     2022.550000
std         1.185005
min      2021.000000
25%      2021.000000
50%      2023.000000
75%      2024.000000
max      2024.000000
Name: purchase_year, dtype: float64

Unique Purchase Years:
[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

--- ID Formatting Check ---
All bike_id values confor

From the output of the bikes table, I noticed that the bike_type seems to have some inconsistency in the formating like the use of capital letters.
Next I want to check for the consistency in the table trips

In [ ]:
print("=== Checking 'trips' Table Consistency ===")
trips_df = tables['trips']

# 1. Inspect data types and structure
print("\n--- Data Types & Structural Info ---")
print(trips_df.info())

# 2. Check for unique categories and casing in 'user_type'
print("\n--- Unique User Types ---")
print(trips_df['user_type'].value_counts(dropna=False))

# 3. Check values and range for 'duration_min'
print("\n--- Duration Statistics ---")
print(trips_df['duration_min'].describe())
negative_durations = trips_df[trips_df['duration_min'] < 0]
print(f"Trips with negative durations: {len(negative_durations)}")

# 4. Check for mixed datetime string formats in 'start_time'
# Let's see if some dates use '/' while others use '-'
slash_dates = trips_df['start_time'].astype(str).str.contains('/')
dash_dates = trips_df['start_time'].astype(str).str.contains('-')
print("\n--- Datetime Formatting Check ---")
print(f"Dates containing '-': {dash_dates.sum()}")
print(f"Dates containing '/': {slash_dates.sum()}")

# Show samples of each format
if slash_dates.any():
    print("\nExample of slash-formatted date:")
    print(trips_df.loc[slash_dates, 'start_time'].head(2).values)
if dash_dates.any():
    print("\nExample of dash-formatted date:")
    print(trips_df.loc[dash_dates, 'start_time'].head(2).values)

=== Checking 'trips' Table Consistency ===

--- Data Types & Structural Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   trip_id           612 non-null    object 
 1   bike_id           612 non-null    object 
 2   start_station_id  612 non-null    object 
 3   end_station_id    612 non-null    object 
 4   start_time        612 non-null    object 
 5   duration_min      582 non-null    float64
 6   user_type         612 non-null    object 
dtypes: float64(1), object(6)
memory usage: 33.6+ KB
None

--- Unique User Types ---
user_type
Member     331
Casual     190
member      35
MEMBER      28
CASUAL      11
Casual       9
casual       8
Name: count, dtype: int64

--- Duration Statistics ---
count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        

From the output, I see the inconsistency in the formating of the user_type.

I also noticed that the statistic of the duramin_min seems unsual. The maximun of the duration seems to be too high (2880) so I suspected it to be an error from data recording. I also think that is why the standard deviation for this data seems unsually high.

Lastly the formating for datetime column is inconsistent.

Next let check for the data consistency for table stations

In [ ]:
print("=== Checking 'stations' Table Consistency ===")
stations_df = tables['stations']

# 1. Inspect data types and structure
print("\n--- Data Types & Info ---")
print(stations_df.info())

# 2. Check unique zones and capitalization issues
print("\n--- Unique Zones ---")
print(stations_df['zone'].value_counts(dropna=False))

# 3. Check capacities range and check for invalid/negative capacities
print("\n--- Capacity Statistics ---")
print(stations_df['capacity'].describe())

# 4. Check for ID formatting (e.g. starting with S followed by numbers)
print("\n--- ID Formatting Check ---")
non_conforming_station_ids = stations_df[~stations_df['station_id'].astype(str).str.match(r'^S\d+$')]
if non_conforming_station_ids.empty:
    print("All station_id values conform to the standard 'Sxx' format.")
else:
    print("Found non-conforming station_ids:")
    display(non_conforming_station_ids)

=== Checking 'stations' Table Consistency ===

--- Data Types & Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_id    12 non-null     object 
 1   station_name  12 non-null     object 
 2   zone          12 non-null     object 
 3   capacity      11 non-null     float64
dtypes: float64(1), object(3)
memory usage: 516.0+ bytes
None

--- Unique Zones ---
zone
Centre    5
North     2
West      2
South     2
East      1
Name: count, dtype: int64

--- Capacity Statistics ---
count    11.000000
mean     22.909091
std       5.539938
min      14.000000
25%      19.000000
50%      24.000000
75%      27.500000
max      29.000000
Name: capacity, dtype: float64

--- ID Formatting Check ---
All station_id values conform to the standard 'Sxx' format.


The output for the table stations looks clean to me. I will ask AI to summarize the findings we got

complete summary of the findings we have gathered across all three tables during our exploratory data analysis:

1. Missing Values (NAs)
trips: Contains 30 missing values in the duration_min column.
stations: Contains 1 missing value in the capacity column (specifically for station S09 - Old Town Hall).
bikes: Contains 0 missing values.
2. Duplications
trips: Contains 12 completely duplicate rows (representing 24 rows in total when matching originals with their duplicates) that share identical trip_ids, times, and stations.
bikes & stations: Contain 0 duplicate rows; their primary keys are fully unique.
3. Inconsistencies and Formatting Issues
bikes table (bike_type): Highly inconsistent due to varied spelling, trailing spaces, and capitalization (e.g., Standard, STANDARD, standard, Electric, Electric , electric, and E-bike).
trips table (user_type): Inconsistent casing issues (e.g., Member, member, MEMBER, Casual, Casual , CASUAL, casual).
trips table (start_time): Mixed datetime string representations. Some use dashes (YYYY-MM-DD HH:MM:SS), while others use slashes (DD/MM/YYYY HH:MM).
trips table (duration_min): The maximum duration is unusually high (2,880 minutes or 48 hours), representing a potential extreme outlier or data entry error compared to a 75th percentile of just 22 minutes.
4. Clean Data Structure
stations: Other than the single missing capacity, this table is fully consistent. The zone categories (Centre, North, West, South, East) are clean and uniform, and all station_id values properly follow the standard Sxx format.
bikes: All bike_id values are correctly formatted as Bxxxx.


## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

I will start the cleaning by removing the NA and duplicate frist. I decided to remove the NA because I cannot recalculate the data based on the input i have.

In [ ]:
# Clean duplicates and missing values in the 'trips' table
trips_df = tables['trips']

# 1. Drop completely duplicate rows
initial_rows = len(trips_df)
trips_df = trips_df.drop_duplicates()
dropped_dupes = initial_rows - len(trips_df)
print(f"Dropped {dropped_dupes} duplicate rows.")

# 2. Drop rows with missing values in 'duration_min'
pre_na_rows = len(trips_df)
trips_df = trips_df.dropna(subset=['duration_min'])
dropped_nas = pre_na_rows - len(trips_df)
print(f"Dropped {dropped_nas} rows with missing duration_min values.")

# Update the global tables dictionary
tables['trips'] = trips_df
print(f"Cleaned 'trips' table now has {len(trips_df)} rows.")

Dropped 12 duplicate rows.
Dropped 30 rows with missing duration_min values.
Cleaned 'trips' table now has 570 rows.


I want to check the NA and duplication after the clean to verify that we actually remove it all

In [ ]:
# Re-register the modified pandas DataFrames back into DuckDB so SQL queries match our cleaned data
for name, df in tables.items():
    con.register(name, df)

for name, df in tables.items():
    print(f"=== Duplication Check in '{name}' table ===")
    # Check for complete row duplicates
    row_dupes = df.duplicated().sum()
    print(f"Complete duplicate rows: {row_dupes}")

    # Check if the primary ID column contains duplicates
    id_col = f"{name[:-1]}_id" if name != "trips" else "trip_id"
    if id_col in df.columns:
        id_dupes = df.duplicated(subset=[id_col]).sum()
        print(f"Duplicate values in primary key '{id_col}': {id_dupes}")
    print()

=== Duplication Check in 'trips' table ===
Complete duplicate rows: 0
Duplicate values in primary key 'trip_id': 0

=== Duplication Check in 'bikes' table ===
Complete duplicate rows: 0
Duplicate values in primary key 'bike_id': 0

=== Duplication Check in 'stations' table ===
Complete duplicate rows: 0
Duplicate values in primary key 'station_id': 0



In [ ]:
for name, df in tables.items():
    print(f"=== NA Counts in '{name}' table ===")
    print(df.isna().sum())
    print()

=== NA Counts in 'trips' table ===
trip_id             0
bike_id             0
start_station_id    0
end_station_id      0
start_time          0
duration_min        0
user_type           0
dtype: int64

=== NA Counts in 'bikes' table ===
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

=== NA Counts in 'stations' table ===
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64



There are still some NA in the table stations. I will ask AI to show me the value for that record

In [ ]:
# Show rows in the stations table where capacity is missing
stations_na = tables['stations'][tables['stations']['capacity'].isna()]
display(stations_na)

,station_id,station_name,zone,capacity
8,S09,Old Town Hall,Centre,NaN


Check whether there are data for this station already. If yes then we remove this record.

In [ ]:
# Pull and display the row for station S09 from the stations table
station_s09 = tables['stations'][tables['stations']['station_id'] == 'S09']
display(station_s09)

,station_id,station_name,zone,capacity
8,S09,Old Town Hall,Centre,NaN


I will keep this record since it is the only record for this station id

I will quickly move on to clean the formating for user_type and bike_type

In [ ]:
# 1. Standardize 'bike_type' in the 'bikes' table
bikes_clean = tables['bikes'].copy()
bikes_clean['bike_type'] = bikes_clean['bike_type'].str.strip().str.lower().replace({
    'standard': 'Standard',
    'electric': 'Electric',
    'e-bike': 'Electric'
})
tables['bikes'] = bikes_clean

# 2. Standardize 'user_type' in the 'trips' table
trips_clean = tables['trips'].copy()
trips_clean['user_type'] = trips_clean['user_type'].str.strip().str.capitalize()
tables['trips'] = trips_clean

# 3. Re-register the cleaned DataFrames in DuckDB
for name, df in tables.items():
    con.register(name, df)

# Verify the clean up
print("Standardized unique bike types:", tables['bikes']['bike_type'].unique())
print("Standardized unique user types:", tables['trips']['user_type'].unique())

Standardized unique bike types: ['Standard' 'Electric']
Standardized unique user types: ['Casual' 'Member']


I will test the data again later due to the time limit

I will prioritising clean the data-time format first

In [ ]:
# Clean and standardize 'start_time' column in the trips table
trips_clean = tables['trips'].copy()

# Convert start_time to datetime with mixed formats
trips_clean['start_time'] = pd.to_datetime(trips_clean['start_time'], format='mixed', dayfirst=True)

# Update the global tables dictionary
tables['trips'] = trips_clean

# Re-register with DuckDB
con.register('trips', trips_clean)

# Verify the change by displaying the data types and a few rows
print("Data type of start_time:", trips_clean['start_time'].dtype)
display(trips_clean[['trip_id', 'start_time']].head(10))

Data type of start_time: datetime64[ns]


,trip_id,start_time
0,T0001,2025-03-01 05:52:00
1,T0002,2025-03-01 06:31:00
2,T0003,2025-03-01 07:06:00
3,T0004,2025-03-01 07:06:00
4,T0005,2025-03-01 07:36:00
5,T0006,2025-03-01 08:06:00
6,T0007,2025-03-01 09:26:00
7,T0008,2025-03-01 10:35:00
8,T0009,2025-03-01 12:14:00
9,T0010,2025-03-01 13:04:00


I will come back to check the output again

I will workon cleaning the outlier for duration minute first. I will check for all the values for duration_min more than 1000

In [ ]:
# Retrieve and display trip records with a duration greater than 1000 minutes
long_trips = tables['trips'][tables['trips']['duration_min'] > 1000]
display(long_trips)

,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member


I will check the statistic of the data after i remove the duramin_min > 1000

In [ ]:
# Remove trips with duration greater than 1000 minutes and print updated statistics
trips_no_outliers = tables['trips'][tables['trips']['duration_min'] <= 1000]

# Save the filtered dataframe back into our tables dictionary
tables['trips'] = trips_no_outliers

# Re-register with DuckDB
con.register('trips', tables['trips'])

# Display the new summary statistics for duration_min
print("=== Summary Statistics for duration_min (No Outliers) ===")
display(tables['trips']['duration_min'].describe())
print(f"Remaining rows in trips table: {len(tables['trips'])}")

=== Summary Statistics for duration_min (No Outliers) ===


,duration_min
count,567.000000
mean,18.044092
std,9.011369
min,4.000000
25%,12.000000
50%,16.000000
75%,22.000000
max,90.000000


Remaining rows in trips table: 567


In [ ]:
# Show updated summary statistics for trip duration after outlier removal
display(tables['trips']['duration_min'].describe())

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [ ]:
# 1. Join trips with stations (on start_station_id) and bikes (on bike_id)
# Note: We use the cleaned data saved in tables['trips'] and tables['bikes']
final_df = tables['trips'].merge(
    tables['stations'], left_on='start_station_id', right_on='station_id', how='inner'
).merge(
    tables['bikes'], on='bike_id', how='inner'
)

# 2. Aggregate to find the number of trips and the average duration in minutes
summary_report = final_df.groupby(['bike_type', 'zone']).agg(
    number_of_trips=('trip_id', 'count'),
    average_duration_min=('duration_min', 'mean')
).reset_index()

# Round average duration for a clean final presentation
summary_report['average_duration_min'] = summary_report['average_duration_min'].round(2)

# Display the report
display(summary_report)

,bike_type,zone,number_of_trips,average_duration_min
0,Electric,Centre,81,14.51
1,Electric,East,11,15.73
2,Electric,North,23,15.96
3,Electric,South,24,13.42
4,Electric,West,22,15.68
5,Standard,Centre,215,18.93
6,Standard,East,20,19.50
7,Standard,North,39,20.36
8,Standard,South,59,19.47
9,Standard,West,73,19.82


### What one row represents:
Each row in this summary table represents a unique combination of a **bike type** (Electric or Standard) and the geographic **zone** (Centre, East, North, South, West) where the trip began. It shows the total count of completed valid trips (`number_of_trips`) and their average travel duration in minutes (`average_duration_min`) for that specific group.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

check the table for NA and duplicates again

In [ ]:
# 1. Verify duplicates in the final cleaned 'trips' table
duplicates_count = tables['trips'].duplicated().sum()
print(f"Remaining complete duplicate rows in 'trips': {duplicates_count}")

# 2. Verify missing values in key columns of the final cleaned 'trips' table
missing_durations = tables['trips']['duration_min'].isna().sum()
print(f"Remaining missing 'duration_min' values in 'trips': {missing_durations}")

# 3. Check for any missing values across all tables to be fully transparent
for name, df in tables.items():
    print(f"\n=== Missing values in '{name}' table ===")
    print(df.isna().sum())

Remaining complete duplicate rows in 'trips': 0
Remaining missing 'duration_min' values in 'trips': 0

=== Missing values in 'trips' table ===
trip_id             0
bike_id             0
start_station_id    0
end_station_id      0
start_time          0
duration_min        0
user_type           0
dtype: int64

=== Missing values in 'bikes' table ===
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

=== Missing values in 'stations' table ===
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64


There are 1 NA in the stations which is expected

Next check for the formating and consistency for all 3 tables

In [ ]:
# 1. Verify standardized values in 'bikes' table
print("=== 'bikes' Table Check ===")
print("Unique bike types (should only contain 'Standard', 'Electric'):")
print(tables['bikes']['bike_type'].unique())

# 2. Verify standardized values in 'trips' table
print("\n=== 'trips' Table Check ===")
print("Unique user types (should only contain 'Member', 'Casual'):")
print(tables['trips']['user_type'].unique())

print("\nParsed 'start_time' datatype (should be datetime64[ns]):")
print(tables['trips']['start_time'].dtype)

# 3. Verify 'stations' table zones
print("\n=== 'stations' Table Check ===")
print("Unique zones:")
print(tables['stations']['zone'].unique())

=== 'bikes' Table Check ===
Unique bike types (should only contain 'Standard', 'Electric'):
['Standard' 'Electric']

=== 'trips' Table Check ===
Unique user types (should only contain 'Member', 'Casual'):
['Casual' 'Member']

Parsed 'start_time' datatype (should be datetime64[ns]):
datetime64[ns]

=== 'stations' Table Check ===
Unique zones:
['Centre' 'North' 'West' 'South' 'East']


Check the stats for the column duration_min

In [ ]:
# Show final statistics for duration_min in the cleaned trips table
display(tables['trips']['duration_min'].describe())

,duration_min
count,567.000000
mean,18.044092
std,9.011369
min,4.000000
25%,12.000000
50%,16.000000
75%,22.000000
max,90.000000


## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*